<!-- MIGRATED_V2_TO_V3_NOTICE -->
> **ℹ️ This notebook is migrated from SageMaker Python SDK v2 to v3.**
>
> This is the v3-based version, and we recommend referring to and using this version. The SageMaker Python SDK v2 and v3 are **not backward compatible**, so v2 code will not run on a v3 installation.
>
> If you are looking for the original v2 version of this notebook, please go to the `v2-archive` branch and look for the notebook with the same name.


# Heterogeneous Cluster - a hello world training job


---

This notebook's CI test result for us-west-2 is as follows. CI test results in other regions can be found at the end of the notebook. 

![This us-west-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-west-2/build_and_train_models|sm-heterogeneous_clusters_training|sm-heterogeneous_clusters_training.ipynb)

---


This basic example shows how to run a Heterogeneous Clusters training job consisting of two instance groups. Each instance group includes a different instance type. Each instance prints its environment information including its instance group and exits.

This notebook uses the SageMaker Python SDK **v3** `ModelTrainer` interface. Heterogeneous clusters are configured through the `Compute` configuration object by passing a list of `InstanceGroup` objects.

You can retrieve environment information in either of the following ways:
  - **Option 1**: Read instance group information using the convenient `sagemaker_training.environment.Environment` class.
  - **Option 2**: Read instance group information from `/opt/ml/input/config/resourceconfig.json`.
 
 
Note: This notebook does not demonstrate offloading of data preprocessing job to data group and deep neural network training to dnn_group. We will cover those examples in [TensorFlow's tf.data.service based Amazon SageMaker Heterogeneous Clusters for training](../tf.data.service.sagemaker/hetero-tensorflow-restnet50.ipynb) and [PyTorch and gRPC distributed dataloader based Amazon SageMaker Heterogeneous Clusters for training](../pt.grpc.sagemaker/hetero-pytorch-mnist.ipynb) notebooks.

### A. Setting up SageMaker Studio notebook
#### Before you start
Ensure you have selected Python 3 image for your SageMaker Studio Notebook instance, and running on _ml.t3.medium_ instance type.

#### Step 1 - Install the SageMaker Python SDK v3 and dependent packages
Heterogeneous Clusters for Amazon SageMaker model training requires the SageMaker Python SDK v3 and up-to-date boto3 client libraries.

In [ ]:
%%bash
python3 -m pip install --upgrade boto3 botocore awscli 'sagemaker>=3.0'

#### Step 2 - Restart the notebook kernel 

In [ ]:
# import IPython
# IPython.Application.instance().kernel.do_shutdown(True)

#### Step 3 - Validate SageMaker Python SDK version
Ensure the output of the cell below reflects:

- SageMaker Python SDK version 3.0.0 or above, 
- boto3 1.24 or above 
- botocore 1.27 or above 

In [ ]:
!pip show sagemaker boto3 botocore |egrep 'Name|Version|---'

### B. Run a heterogeneous cluster training job

#### Step 1: Set up training environment
Import the required libraries that enable you to use Heterogeneous clusters for training. In this step, you are also inheriting this notebook's IAM role and SageMaker session. 

In [ ]:
import datetime

from sagemaker.train import ModelTrainer
from sagemaker.train.configs import SourceCode, Compute
from sagemaker.core.shapes import InstanceGroup, StoppingCondition
from sagemaker.core.helper.session_helper import Session, get_execution_role
from sagemaker.core import image_uris

sess = Session()
role = get_execution_role()
region = sess.boto_region_name

#### Step 2: Define instance groups 
Here we define instance groups. Each instance group includes a different instance type.

In [ ]:
data_group = InstanceGroup(
    instance_group_name="data_group", instance_type="ml.c5.xlarge", instance_count=1
)
dnn_group = InstanceGroup(
    instance_group_name="dnn_group", instance_type="ml.m4.xlarge", instance_count=1
)

#### Step 3: Review the "hello world" training code

In [ ]:
!pygmentize source_dir/train.py

#### Step 4: Configure the ModelTrainer
In order to use SageMaker to run our training code, we create a `ModelTrainer` that defines how to use the container to train. In SageMaker Python SDK v3, framework estimators (such as the v2 `TensorFlow` estimator) are replaced by the generic `ModelTrainer` combined with a framework training image retrieved via `image_uris.retrieve`.

The heterogeneous cluster is configured through the `Compute` object by passing the list of instance groups. Note that when `instance_groups` is provided, you do not set `instance_type`/`instance_count` on `Compute`.

In [ ]:
# Retrieve the TensorFlow training image (v2 used framework_version="2.9", py_version="py39")
training_image = image_uris.retrieve(
    framework="tensorflow",
    region=region,
    version="2.9",
    py_version="py39",
    instance_type="ml.c5.xlarge",
    image_scope="training",
)

source_code = SourceCode(
    source_dir="./source_dir",
    entry_script="train.py",
)

compute = Compute(
    instance_groups=[
        data_group,
        dnn_group,
    ],
    volume_size_in_gb=10,
)

model_trainer = ModelTrainer(
    training_image=training_image,
    source_code=source_code,
    compute=compute,
    stopping_condition=StoppingCondition(max_runtime_in_seconds=3600),
    role=role,
    base_job_name="hello-world-heterogenous",
)

#### Step 5: Submit the training job
Here you are submitting the heterogeneous cluster training job. 

In [ ]:
model_trainer.train()

#### Step 6: Review the logs for environment information

Wait for the training job to finish, and review its logs in the AWS Console (click on **View logs** from the **Training Jobs** node in **Amazon SageMaker Console**)  You'll find two logs: Algo1, Algo2. Examine the printouts on each node on how to retrieve instance group environment information. An example is shown here:

```
Option-1: Read instance group information from the sagemaker_training.environment.Environment class
env.is_hetero: True
env.current_host: algo-1
env.current_instance_type: ml.c5.xlarge
env.current_instance_group: data_group
env.current_instance_group_hosts: ['algo-1']
env.instance_groups: ['data_group', 'dnn_group']

Option-2: Read instance group information from {file_path}.            You'll need to parse the json yourself. This doesn't require an additional library.
/opt/ml/input/config/resourceconfig.json dump = {
    "current_group_name": "data_group",
    "current_host": "algo-1",
    "current_instance_type": "ml.c5.xlarge",
    "hosts": [
        "algo-1",
        "algo-2"
    ],
    "instance_groups": [
        {
            "hosts": [
                "algo-1"
            ],
            "instance_group_name": "data_group",
            "instance_type": "ml.c5.xlarge"
        },
        {
            "hosts": [
                "algo-2"
            ],
            "instance_group_name": "dnn_group",
            "instance_type": "ml.m4.xlarge"
        }
    ],
    "network_interface_name": "eth0"
}
env.is_hetero: True
current_host=algo-1
current_instance_type=ml.c5.xlarge
env.current_instance_group: data_group
env.current_instance_group_hosts: TODO
env.instance_groups: TODO
env.instance_groups_dict: [{'instance_group_name': 'data_group', 'instance_type': 'ml.c5.xlarge', 'hosts': ['algo-1']}, {'instance_group_name': 'dnn_group', 'instance_type': 'ml.m4.xlarge', 'hosts': ['algo-2']}]
env.distribution_hosts: TODO
env.distribution_instance_groups: TODO
```

### C. Next steps

In this notebook, we demonstrated how to retrieve the environment information, and differentiate which instance group an instance belongs to. Based on this, you can build logic to offload data processing tasks in your training job to a dedicated instance group. To understand how that can be done with a real-world example, we suggest going through the following notebook examples:  

- [TensorFlow's tf.data.service based Amazon SageMaker Heterogeneous Clusters for training](../tf.data.service.sagemaker/hetero-tensorflow-restnet50.ipynb)
- [PyTorch and gRPC distributed dataloader based Amazon SageMaker Heterogeneous Clusters for training](../pt.grpc.sagemaker/hetero-pytorch-mnist.ipynb)

## Notebook CI Test Results

This notebook was tested in multiple regions. The test results are as follows, except for us-west-2 which is shown at the top of the notebook.

![This us-east-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-east-1/build_and_train_models|sm-heterogeneous_clusters_training|sm-heterogeneous_clusters_training.ipynb)

![This us-east-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-east-2/build_and_train_models|sm-heterogeneous_clusters_training|sm-heterogeneous_clusters_training.ipynb)

![This us-west-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-west-1/build_and_train_models|sm-heterogeneous_clusters_training|sm-heterogeneous_clusters_training.ipynb)

![This ca-central-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ca-central-1/build_and_train_models|sm-heterogeneous_clusters_training|sm-heterogeneous_clusters_training.ipynb)

![This sa-east-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/sa-east-1/build_and_train_models|sm-heterogeneous_clusters_training|sm-heterogeneous_clusters_training.ipynb)

![This eu-west-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-west-1/build_and_train_models|sm-heterogeneous_clusters_training|sm-heterogeneous_clusters_training.ipynb)

![This eu-west-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-west-2/build_and_train_models|sm-heterogeneous_clusters_training|sm-heterogeneous_clusters_training.ipynb)

![This eu-west-3 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-west-3/build_and_train_models|sm-heterogeneous_clusters_training|sm-heterogeneous_clusters_training.ipynb)

![This eu-central-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-central-1/build_and_train_models|sm-heterogeneous_clusters_training|sm-heterogeneous_clusters_training.ipynb)

![This eu-north-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-north-1/build_and_train_models|sm-heterogeneous_clusters_training|sm-heterogeneous_clusters_training.ipynb)

![This ap-southeast-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-southeast-1/build_and_train_models|sm-heterogeneous_clusters_training|sm-heterogeneous_clusters_training.ipynb)

![This ap-southeast-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-southeast-2/build_and_train_models|sm-heterogeneous_clusters_training|sm-heterogeneous_clusters_training.ipynb)

![This ap-northeast-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-northeast-1/build_and_train_models|sm-heterogeneous_clusters_training|sm-heterogeneous_clusters_training.ipynb)

![This ap-northeast-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-northeast-2/build_and_train_models|sm-heterogeneous_clusters_training|sm-heterogeneous_clusters_training.ipynb)

![This ap-south-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-south-1/build_and_train_models|sm-heterogeneous_clusters_training|sm-heterogeneous_clusters_training.ipynb)
